# Walmart Sales Data Cleaning

**Author:** Bintang Fausta Listianto
**Date:** 07/23/2026

## Overview

Notebook ini berisi proses **data cleaning** untuk dataset penjualan Walmart (`Walmart.csv`) menggunakan Python dan pandas. Data mentah memiliki beberapa masalah seperti missing values, duplikasi baris, tipe data yang tidak sesuai, dan format kolom yang tidak konsisten.

Setelah dibersihkan, data diekspor ke `walmart_clean_data.csv` dan dimuat ke database PostgreSQL untuk tahap selanjutnya.

> **Catatan:** Notebook ini fokus khusus pada tahap *data cleaning*. Proses *Exploratory Data Analysis* (EDA) dan analisis bisnis dilakukan secara terpisah menggunakan **SQL** — lihat folder `/sql` pada repository ini.

**Tools yang digunakan:**
- `pandas` — manipulasi dan pembersihan data
- `sqlalchemy` + `psycopg2` — koneksi dan load data ke PostgreSQL
- `python-dotenv` — mengelola kredensial database secara aman

**Dataset:**
- Sumber: `Walmart.csv`
- 10.051 baris, 11 kolom sebelum dibersihkan

---
## 1. Setup & Environment Variables

Mengimpor library yang dibutuhkan. Kredensial database **tidak** ditulis langsung di kode, melainkan disimpan di file `.env` (tidak ikut di-commit ke repo) dan dibaca menggunakan `python-dotenv`. Contoh format variabel tersedia di `.env.example`.

In [1]:
#importing dependencies

import pandas as pd

#psql toolkit
import psycopg2 #this will work as adapter
from sqlalchemy import create_engine

#mysql
import pymysql

---
## 2. Load Dataset

Memuat data mentah dari file CSV.

In [3]:
df = pd.read_csv('Walmart.csv', encoding_errors='ignore')
df.shape

(10051, 11)

---
## 3. Initial Data Overview

Melihat struktur data, tipe kolom, dan statistik deskriptif sebelum dibersihkan.

In [4]:
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
0,1,WALM003,San Antonio,Health and beauty,$74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48
1,2,WALM048,Harlingen,Electronic accessories,$15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48
2,3,WALM067,Haltom City,Home and lifestyle,$46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33
3,4,WALM064,Bedford,Health and beauty,$58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33
4,5,WALM013,Irving,Sports and travel,$86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48


In [ ]:
# Mengecek kolom manakah yang ada nullnya
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10051 entries, 0 to 10050
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      10051 non-null  int64  
 1   Branch          10051 non-null  str    
 2   City            10051 non-null  str    
 3   category        10051 non-null  str    
 4   unit_price      10020 non-null  str    
 5   quantity        10020 non-null  float64
 6   date            10051 non-null  str    
 7   time            10051 non-null  str    
 8   payment_method  10051 non-null  str    
 9   rating          10051 non-null  float64
 10  profit_margin   10051 non-null  float64
dtypes: float64(3), int64(1), str(7)
memory usage: 863.9 KB


In [ ]:
df.describe()
# Terlihat quantity jumlahnya ada dibawah invoice menandakan ada yang keliru (quantity harusnya lebih banyak)

,invoice_id,quantity,rating,profit_margin
count,10051.000000,10020.000000,10051.000000,10051.000000
mean,5025.741220,2.353493,5.825659,0.393791
std,2901.174372,1.602658,1.763991,0.090669
min,1.000000,1.000000,3.000000,0.180000
25%,2513.500000,1.000000,4.000000,0.330000
50%,5026.000000,2.000000,6.000000,0.330000
75%,7538.500000,3.000000,7.000000,0.480000
max,10000.000000,10.000000,10.000000,0.570000


Dari hasil `describe()`, jumlah data non-null pada kolom `quantity` (10.020) lebih sedikit dibanding `invoice_id` (10.051) — ini mengindikasikan adanya *missing value* pada kolom tersebut yang perlu ditangani di tahap berikutnya.

---
## 4. Standardizing Column Names

Menyamakan seluruh nama kolom menjadi huruf kecil di awal proses, agar konsisten pada seluruh tahap cleaning berikutnya (termasuk saat diekspor ke CSV dan database).

In [ ]:
# Memperbaiki nama kolom menjadi huruf kecil semua
df.columns = df.columns.str.lower()
df.columns

Index(['invoice_id', 'branch', 'city', 'category', 'unit_price', 'quantity',
       'date', 'time', 'payment_method', 'rating', 'profit_margin', 'total'],
      dtype='str')

---
## 5. Checking & Removing Duplicates

In [ ]:
# Menghitung berapa data yang duplikat
df.duplicated().sum()

np.int64(51)

In [8]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

---
## 6. Handling Missing Values

In [ ]:
# Menghitung berapa data yang null
df.isnull().sum()

invoice_id         0
Branch             0
City               0
category           0
unit_price        31
quantity          31
date               0
time               0
payment_method     0
rating             0
profit_margin      0
dtype: int64

In [ ]:
# Menghapus semua baris yang memiliki missing value(Null)
df.dropna(inplace=True)

df.isnull().sum()

invoice_id        0
Branch            0
City              0
category          0
unit_price        0
quantity          0
date              0
time              0
payment_method    0
rating            0
profit_margin     0
dtype: int64

Kolom `unit_price` dan `quantity` masing-masing memiliki 31 missing value (kurang dari 1% dari total data). Karena jumlahnya kecil, baris dengan missing value ini dihapus agar tidak mengganggu perhitungan pada tahap selanjutnya.

---
## 7. Fixing Data Types

Kolom `unit_price` tersimpan sebagai teks dengan simbol `$` (misalnya `"$74.69"`), sehingga perlu dikonversi menjadi tipe numerik.

In [ ]:
# Melihat type data
df.dtypes

invoice_id          int64
Branch                str
City                  str
category              str
unit_price            str
quantity          float64
date                  str
time                  str
payment_method        str
rating            float64
profit_margin     float64
dtype: object

In [ ]:
# Menghapus simbol '$' pada unit_price dan mengubah tipe data menjadi float
df['unit_price'] = df['unit_price'].str.replace('$', '').astype(float)


---
## 8. Feature Engineering: Kolom `total`

Menambahkan kolom `total` yang merepresentasikan total nilai transaksi per baris (`unit_price` × `quantity`), untuk mempermudah analisis penjualan pada tahap SQL berikutnya.

In [17]:
df['total'] = df['unit_price'] * df['quantity']
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,total
0,1,WALM003,San Antonio,Health and beauty,74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48,522.83
1,2,WALM048,Harlingen,Electronic accessories,15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48,76.40
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33,324.31
3,4,WALM064,Bedford,Health and beauty,58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33,465.76
4,5,WALM013,Irving,Sports and travel,86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48,604.17


---
## 9. Exporting Cleaned Data

Menyimpan data yang sudah bersih ke file CSV baru.

In [24]:
df.to_csv('walmart_clean_data.csv', index=False)

---
## 10. Loading to PostgreSQL

Memuat data bersih ke database PostgreSQL untuk digunakan pada tahap analisis SQL. Kredensial database diambil dari environment variable, bukan ditulis langsung di kode.

In [ ]:
engine_postgresql = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

try:
    with engine_postgresql.connect():
        print("Connection successful to PostgreSQL")
except Exception as e:
    print("Unable to connect:", e)

Connection Successed to postgresql


In [34]:
df.to_sql(name='walmart', con=engine_postgresql, if_exists='append', index=False)

969

Memverifikasi jumlah baris pada tabel di database sesuai dengan jumlah baris pada data yang sudah dibersihkan.

In [ ]:
pd.read_sql("SELECT COUNT(*) FROM walmart;", engine_postgresql)


,count
0,9969


---
## Summary

- Data awal: **10.051 baris, 11 kolom**.
- Menghapus **51 baris duplikat**.
- Menghapus **31 baris** dengan missing value pada `unit_price` dan `quantity`.
- Menstandarkan seluruh nama kolom menjadi huruf kecil.
- Memperbaiki tipe data `unit_price` dari teks menjadi numerik.
- Menambahkan kolom baru `total` (`unit_price` × `quantity`).
- Data akhir: **9.969 baris, 12 kolom**, diekspor ke `walmart_clean_data.csv` dan dimuat ke tabel `walmart` pada PostgreSQL.

**Langkah selanjutnya:** Exploratory Data Analysis dan analisis bisnis dilakukan menggunakan SQL. Lihat folder `/sql` pada repository ini untuk query dan insight lebih lanjut.
